# Build Spark Session

In [18]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataReadMalformed")
    .master("local[*]")
    .getOrCreate()
)

spark

## Read Single CSV
### Syntax:
### spark.read.csv('Path')
### sep='delimiter', dateFormat='yyyy-MM-dd',mode='PERMISSIVE/DROPMALFORMED/FAILFAST' 
### **PERMISSIVE** - Capture Malformed Records
### **DROPMALFORMED** - Ignore Malformed Records
### **FAILFAST** - Stop and Raise the Exception

In [19]:
MalformedFile=r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Malfored_Customer_Data_20210101.csv"

customer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(MalformedFile)
)
print("Customer DataFrame with Malformed Data:")
customer_df.show(truncate=False)



Customer DataFrame with Malformed Data:
+----------+-------+------------+
|Customerid|zipCode|PurchaseDate|
+----------+-------+------------+
|AMZ0000006|522436 |2011-01-01  |
|AMZ0000007|522436 |01-02-2011  |
|AMZ0000008|522436 |2011-01-03  |
|AMZ0000009|522436 |2011-01-04  |
|AMZ0000010|KXKDU  |2011-01-05  |
|AMZ0000011|522436 |2011-01-06  |
|AMZ0000012|522436 |2011-01-07  |
+----------+-------+------------+



### PERMISSIVE

In [20]:

from pyspark.sql.types import StructType, StructField, IntegerType, StringType

Malschema = StructType([
    StructField("Customerid", StringType(), True),
    StructField("zipCode", IntegerType(), True),
    StructField("PurchaseDate", StringType(), True),
    StructField("_corrupt_record", StringType(), True)  # Tracks malformed data
])


customer_df = (
    spark.read
    .option("header", True)
    .schema(Malschema)
    .option("mode", "PERMISSIVE")  # Allows malformed records to be captured
    .csv(MalformedFile)
)

print("Defining a zipCode as IntegerType and Mode as permissive to handle Malformed Data:")
customer_df.show(truncate=False)



Defining a zipCode as IntegerType and Mode as permissive to handle Malformed Data:
+----------+-------+------------+---------------------------+
|Customerid|zipCode|PurchaseDate|_corrupt_record            |
+----------+-------+------------+---------------------------+
|AMZ0000006|522436 |2011-01-01  |NULL                       |
|AMZ0000007|522436 |01-02-2011  |NULL                       |
|AMZ0000008|522436 |2011-01-03  |NULL                       |
|AMZ0000009|522436 |2011-01-04  |NULL                       |
|AMZ0000010|NULL   |2011-01-05  |AMZ0000010,KXKDU,2011-01-05|
|AMZ0000011|522436 |2011-01-06  |NULL                       |
|AMZ0000012|522436 |2011-01-07  |NULL                       |
+----------+-------+------------+---------------------------+



### DROPMALFORMED

In [21]:

customer_df = (
    spark.read
    .option("header", True)
    .schema(Malschema)
    .option("mode", "DROPMALFORMED")  # Drops malformed records
    .csv(MalformedFile)
)

print("Defining a zipCode as IntegerType and Mode as DROPMALFORMED to handle Malformed Data:")
customer_df.show(truncate=False)


Defining a zipCode as IntegerType and Mode as DROPMALFORMED to handle Malformed Data:
+----------+-------+------------+---------------+
|Customerid|zipCode|PurchaseDate|_corrupt_record|
+----------+-------+------------+---------------+
|AMZ0000006|522436 |2011-01-01  |NULL           |
|AMZ0000007|522436 |01-02-2011  |NULL           |
|AMZ0000008|522436 |2011-01-03  |NULL           |
|AMZ0000009|522436 |2011-01-04  |NULL           |
|AMZ0000011|522436 |2011-01-06  |NULL           |
|AMZ0000012|522436 |2011-01-07  |NULL           |
+----------+-------+------------+---------------+



### FAILFAST

In [22]:
customer_df = (
    spark.read
    .option("header", True)
    .schema(Malschema)
    .option("mode", "FAILFAST")  # Allows Error if malformed records found.
    .csv(MalformedFile)
)

print("Defining a Mode as FAILFAST to handle Malformed Data:")
customer_df.show(truncate=False)

Defining a Mode as FAILFAST to handle Malformed Data:


Py4JJavaError: An error occurred while calling o232.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///C:/Users/navat/AzureDataBricks/Azure_Databricks/Raw_Source/Customer_Data/Malfored_Customer_Data_20210101.csv.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEva